## 1. The Semantic Scholar Search (S2Search) Reranker

In [1]:
from s2search.rank import S2Ranker

# point to the artifacts downloaded from s3
data_dir = 's2search_data'

# the data is a list of dictionaries
papers = [
    {
        'title': 'Neural Networks are Great',
        'abstract': 'Neural networks are known to be really great models. You should use them.',
        'venue': 'Deep Learning Notions',
        'authors': ['Sergey Feldman', 'Gottfried W. Leibniz'],
        'year': 2019,
        'n_citations': 100,
        'n_key_citations': 10
    },
    {
        'title': 'Neural Networks are Terrible',
        'abstract': 'Neural networks have only barely worked and we should stop working on them.',
        'venue': 'JMLR',
        'authors': ['Isaac Newton', 'Sergey Feldman'],
        'year': 2009,
        'n_citations': 5000  # we don't have n_key_citations here and that's OK
    }
]

# only do this once because we have to load the giant language models into memory
s2ranker = S2Ranker(data_dir)

# higher scores are better
print(s2ranker.score('neural networks', papers))
print(s2ranker.score('jmlr', papers))
print(s2ranker.score('missing', papers))

[7.46771709 7.74104785]
[-15.90676457   5.41275995]
[-15.90676457 -15.59990005]


## 2. Paper Metadata From ArXiv

![alt text](img/arxiv_cate.png)

## 3. XAI Techniqeus

### 3.1 ALE

In [2]:
from xai_process import XaiProcess
dsp = "exp-csar-demo"
xai_process = XaiProcess(dsp)

quantile_config = {
    'year': 10, 'n_citations': 10, 'title': 10,
    'abstract': 10, 'authors': 10, 'venue': 10
}

import ranker_helper as rh
rh.work_load = 1

ale_rs = xai_process.ale_scores()

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

Loading process ranker model...
Load the process s2 ranker within 5.57 sec


In [3]:
print("ALE Scores for each feature in the csar partition")
ale_rs

ALE Scores for each feature in the csar partition


,csar
title,"[-0.47447874641709875, -0.47447874641709875, -..."
abstract,"[-6.918134609921304, -6.918134609921304, -6.91..."
venue,"[-1.2309831116405803, -1.2309831116405803, -1...."
authors,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
year,"[-0.7320250963992377, -0.3988737500728372, -0...."
n_citations,"[-0.326796861357312, -0.326796861357312, -0.29..."


In [4]:
print("Summarized ALE Score: mean of the ALE scores")
xai_process.ale_exp_sum(ale_rs)    

Summarized ALE Score: mean of the ALE scores


,csar
title,0.011957
abstract,0.028459
venue,0.031021
authors,0.000000
year,0.001682
n_citations,0.004269


### 3.2 Shaply Value

In [5]:
sv_rs = xai_process.shapley_value()

Scores of exp-csar-demo_csar_t1_origin.npz exist, should pass


  0%|          | 0/63 [00:00<?, ?it/s]

Done with exp-csar-demo csar


In [6]:
sv_rs['csar']

,title,abstract,venue,authors,year,n_citations
0,0.015451,0.015451,0.015451,-0.108966,0.214238,0.015451
1,0.015451,0.015451,0.015451,-0.112781,0.445563,0.536135
2,26.798314,0.015451,0.015451,-0.056917,0.466220,0.015451
3,0.015451,0.015451,0.015451,-0.108966,0.715472,0.015451
4,0.015451,0.015451,0.015451,-0.108966,0.715472,0.015451
...,...,...,...,...,...,...
2558,0.015451,24.492041,0.015451,-0.093284,0.241442,0.168547
2559,0.015451,24.477894,0.015451,-0.093284,0.486104,0.109176
2560,0.015451,0.015451,0.015451,-0.108966,0.269175,0.015451
2561,0.015451,0.015451,0.015451,-0.105394,0.391149,0.314528


In [7]:
sv_rs['csar'].mean(), sv_rs['csar'].mean().values.argsort()[::-1]

(title          1.702758
 abstract       8.098899
 venue          0.071804
 authors       -0.096348
 year           0.297469
 n_citations    0.151330
 dtype: float64,
 array([1, 0, 4, 5, 2, 3]))

In [8]:
sv_rs['csar'].mean().index[sv_rs['csar'].mean().values.argsort()[::-1]]

Index(['abstract', 'title', 'year', 'n_citations', 'venue', 'authors'], dtype='object')

## 4. Explanation Consistancy Evaluation

In [9]:
data_partition_exps = [
    "exp-csar-demo",
    "exp-cscc-demo",
    "exp-cssd-demo",
    "exp-csse-demo",
]

feature_index = {
    'title': 0,
    'abstract': 1,
    'venue': 2,
    'authors': 3,
    'year': 4,
    'n_citations': 5,
}

feature_importance_order_one_hot_rs = {
    'ale':[],
    'shapley_value':[],
}

for data_partition_exp in data_partition_exps:
    xp = XaiProcess(data_partition_exp)
    data_partition = data_partition_exp.split('-')[1]
    ale_rs = xp.ale_scores()
    ale_exp_sum = xai_process.ale_exp_sum(ale_rs) 
    # display(ale_exp_sum)
    ale_feature_importance_order = ale_exp_sum\
        .sort_values(by=data_partition, ascending=False, inplace=False).index.values.tolist()
    # print(ale_feature_importance_order)
    ale_feature_importance_order_one_hot = [
        feature_index[f]
        for f in ale_feature_importance_order
    ]
    feature_importance_order_one_hot_rs['ale'].append(ale_feature_importance_order_one_hot)

    sv_rs = xp.shapley_value()[data_partition].mean()
    sv_feature_importance_order = \
        sv_rs.index[sv_rs.values.argsort()[::-1]]
    sv_feature_importance_order_one_hot = [
        feature_index[f]
        for f in sv_feature_importance_order
    ]
    feature_importance_order_one_hot_rs['shapley_value'].append(
        sv_feature_importance_order_one_hot)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

Scores of exp-csar-demo_csar_t1_origin.npz exist, should pass


  0%|          | 0/63 [00:00<?, ?it/s]

Done with exp-csar-demo csar


...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

Scores of exp-csse-demo_csse_t1_origin.npz exist, should pass


  0%|          | 0/63 [00:00<?, ?it/s]

Done with exp-csse-demo csse


In [10]:
feature_importance_order_one_hot_rs

{'ale': [[2, 1, 0, 5, 4, 3],
  [0, 2, 1, 5, 4, 3],
  [2, 1, 0, 5, 4, 3],
  [2, 1, 0, 3, 5, 4]],
 'shapley_value': [[1, 0, 4, 5, 2, 3],
  [1, 0, 2, 5, 4, 3],
  [1, 0, 4, 5, 2, 3],
  [1, 2, 0, 4, 5, 3]]}

In [11]:
def normalised_kendall_tau_distance(values1, values2):
    """Compute the Kendall tau distance."""
    n = len(values1)
    assert len(values2) == n, "Both lists have to be of equal length"
    i, j = np.meshgrid(np.arange(n), np.arange(n))
    a = np.argsort(values1)
    b = np.argsort(values2)
    ndisordered = np.logical_or(np.logical_and(a[i] < a[j], b[i] > b[j]), np.logical_and(a[i] > a[j], b[i] < b[j])).sum()
    return ndisordered / (n * (n - 1))

In [14]:
for xai, feature_importance_orders in feature_importance_order_one_hot_rs.items():
    # print("")
    consistency_per_xai = []
    for i in range(4):
        for j in range(1,4):
            d = normalised_kendall_tau_distance(feature_importance_orders[i], feature_importance_orders[j])
            consistency_per_xai.append(d)
    print(f"Feature Importance Explanation Consistency Evaluation of {xai}: {np.array(consistency_per_xai).mean()}")

Feature Importance Explanation Consistency Evaluation of ale: 0.1111111111111111
Feature Importance Explanation Consistency Evaluation of shapley_value: 0.12222222222222223
